# Inferência com TFLite quantizado (passo a passo)
Este notebook demonstra como usar um modelo TFLite (int8/dynamic/fp16) com o mesmo pipeline de features/normalização usado no projeto.

In [ ]:
# Configuração de ambiente
import os
os.environ['MODEL_BACKEND'] = 'tflite'
# Ajuste o caminho para seu .tflite exportado
os.environ['TFLITE_MODEL_PATH'] = r'tflite_exports\model_int8.tflite'
print('MODEL_BACKEND =', os.getenv('MODEL_BACKEND'))
print('TFLITE_MODEL_PATH =', os.getenv('TFLITE_MODEL_PATH'))

In [ ]:
# Imports e pipeline de features
import sys, os
sys.path.append(os.path.abspath('./finance/AI/Processing/'))
from DataLoaderPipeline import FeaturesDataGenerator, scrapingHistoricalData
import numpy as np
from datetime import datetime, timedelta
import json
from pathlib import Path

In [ ]:
# Parâmetros do modelo (carregar de config.json se disponível)
MODEL_DIR = Path(r'finance/AI/Classification/Experiments/Cryptos/models/model_CNN_MultiHead_2D_crypto_BTC_last')  # ajuste conforme seu modelo
try:
    with open(MODEL_DIR / 'config.json', 'r', encoding='utf-8') as f:
        parameters = json.load(f)
except Exception:
    parameters = {
        'datatype': '2D', 'lookback': 64, 'pred_days': 0, 'shuffle': False, 'batch_size': 32,
        'features_indicators': ['EMA','MACD','RSI','BBANDS'], 'data_augmentation': False,
        'min_norm': 0.0, 'max_norm': 1.0, 'TH': [0.5,0.5,0.5]
    }
parameters

In [ ]:
# Coleta dados e gera x_float
SHD = scrapingHistoricalData()
symbol = 'BTC'
interval = '4h'
start_time = (datetime.today() - timedelta(days=60)).strftime('%Y-%m-%d')
data_df = SHD.get_crypto_historical_data([symbol], interval, start_time)
dataGen = FeaturesDataGenerator(
    data_df,
    datatype=parameters['datatype'],
    lookback=parameters['lookback'],
    pred_days=parameters['pred_days'],
    shuffle=False,
    batch_size=32,
    selected_features=parameters['features_indicators'],
    data_augmentation=False,
    min_max_norm_features=[parameters['min_norm'], parameters['max_norm']],
)
x_inf = dataGen.comput_features(data_df, pred_days=0)
x_float = dataGen.apply_NomrMinmax(x_inf, parameters['min_norm'], parameters['max_norm'], axis=0)
if parameters['datatype'] == '2D':
    x_float = np.transpose(x_float, [0, 2, 1]).reshape(-1, dataGen.inputShape[1], dataGen.inputShape[2], 1)
x_float.shape, x_float.dtype

In [ ]:
# Wrapper TFLite para predição
import tensorflow as tf
import numpy as np
from pathlib import Path

class TFLiteModel:
    """Wrapper simples para inferência TFLite (int8/float)."""
    def __init__(self, tflite_path):
        self.interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
        self.interpreter.allocate_tensors()
        self.input_details = self.interpreter.get_input_details()
        self.output_details = self.interpreter.get_output_details()
        self.input_index = self.input_details[0]['index']
        self.output_index = self.output_details[0]['index']
        self.input_scale, self.input_zero_point = self.input_details[0].get('quantization', (0.0, 0))
        self.output_scale, self.output_zero_point = self.output_details[0].get('quantization', (0.0, 0))
        self.input_dtype = self.input_details[0]['dtype']
        self.output_dtype = self.output_details[0]['dtype']
        self.input_shape = self.input_details[0]['shape']
    def _ensure_shape(self, x):
        if tuple(x.shape) != tuple(self.input_shape):
            self.interpreter.resize_tensor_input(self.input_index, x.shape)
            self.interpreter.allocate_tensors()
    def predict_proba(self, x_float):
        if x_float.ndim == 3:
            x_float = x_float[..., np.newaxis]
        outs = []
        for i in range(x_float.shape[0]):
            x_in = x_float[i:i+1]
            self._ensure_shape(x_in)
            if self.input_dtype == np.int8:
                x_cast = (x_in / self.input_scale + self.input_zero_point).astype(np.int8)
            else:
                x_cast = x_in.astype(self.input_dtype)
            self.interpreter.set_tensor(self.input_index, x_cast)
            self.interpreter.invoke()
            y = self.interpreter.get_tensor(self.output_index)
            if self.output_dtype == np.int8:
                y = (y.astype(np.float32) - self.output_zero_point) * self.output_scale
            else:
                y = y.astype(np.float32)
            outs.append(y[0])
        return np.stack(outs, axis=0)


In [ ]:
# Executa predição TFLite
tflite_path = Path(os.getenv('TFLITE_MODEL_PATH'))
tm = TFLiteModel(tflite_path)
preds = tm.predict_proba(x_float)
preds.shape, preds[:2]

In [ ]:
# Geração de sinais simples
trade = ['Hold','Buy','Sell']
TH = parameters.get('TH', [0.5,0.5,0.5])
signals = [ trade[np.argmax(p)] if np.max(p) > TH[np.argmax(p)] else trade[0] for p in preds ]
signals[-5:]

Resumo: configurou ambiente, gerou `x_float`, executou TFLite e produziu sinais básicos.